In [ ]:
from pathlib import Path
import json
curr = Path().resolve()  # this is the notebook dir
for parent in [curr] + list(curr.parents):
    candidate = parent / "filepaths.json"
    if candidate.exists():
        with open(candidate) as f:
            FILEPATHS = json.load(f)

import os
os.chdir(FILEPATHS["Bill Calculator"]["parent"])

import warnings
warnings.filterwarnings("ignore")
from itertools import cycle
import importlib
import genability_cost
importlib.reload(genability_cost)
from genability_cost import genability_costs
import genability_cost_copy
importlib.reload(genability_cost_copy)
from genability_cost_copy import genability_costs_hack

elecTariff = ("2606","2614")
gasTariff = ("R-3-RESIDENTIAL HEATING---", "R-1-RESIDENTIAL NON-HEATING---")
state = "MA"
utility = "NSTAR Electric Company"
gas_utility = "Eversource Energy (NStar) Eastern"

# Open below for tariff and utility details
"""
elecTariff: Required. Use the masterTariffId from Genability. Give as tuple. If one give it as (x,).
gasTariff: Required. Leave blank if you don't know it, it will give you suggestions. Use exact RateActuity name when you know it.
           Provide as a tuple of heating and non-heating rate in that order. Some states have a separate non-heating rate for non-gas heating.
           If they don't just put one tariff as a tuple (x,).
state: Required. Use the state abbreviation (e.g., "CA", "TX").
utility: Required. Use any utility name based on EIA name, it will give you suggestions if not valid.
gas_utility: Required. If you leave this blank, or if your input is invalid, it will give options.
"""

upgrade = [3,]

# Open below for upgrade options
"""
Give as list. If one give it as [x,]. If you do this it will calculate costs for upgraded
load profiles in addition to base profiles. Don't include baseline 0 in list.
  0: "Baseline",
  1: "ENERGY STAR heat pump with elec backup",
  2: "High efficiency cold-climate heat pump with elec backup",
  3: "Ultra high efficiency heat pump with elec backup",
  4: "ENERGY STAR heat pump with existing system as backup",
  5: "Geothermal heat pump",
  6: "ENERGY STAR heat pump with elec backup + Light Touch Envelope",
  7: "High efficiency cold-climate heat pump with elec backup + Light Touch Envelope",
  8: "Ultra high efficiency heat pump with elec backup + Light Touch Envelope",
  9: "ENERGY STAR heat pump with existing system as backup + Light Touch Envelope",
  10: "Geothermal heat pump + Light Touch Envelope",
  11: "ENERGY STAR heat pump with elec backup + Light Touch Envelope + Full Appliance Electrification with Efficiency",
  12: "High efficiency cold-climate heat pump with elec backup + Light Touch Envelope + Full Appliance Electrification with Efficiency",
  13: "Ultra high efficiency heat pump with elec backup + Light Touch Envelope + Full Appliance Electrification with Efficiency",
  14: "ENERGY STAR heat pump with existing system as backup + Light Touch Envelope + Full Appliance Electrification with Efficiency",
  15: "Geothermal heat pump + Light Touch Envelope + Full Appliance Electrification with Efficiency",
  16: "Envelope Only - Light Touch Envelope"
"""

# Any combination of these is fine, no need to fill them all
segment = {
    "heating_type":         "Natural Gas",
    "building_type":        "SF",
    "area":                 "",
    "income":               "",
    "climate_zone":         "",
    "heating_efficiency":   "",
    "cooling_type":         "",
    "vintage":              "",
    "insulation_level":     "",
    "has_solar":            ""
}

# Open below for segment options
"""
heating_type: Electric HP | Electric Resistance | Natural Gas | Propane | Other
building_type: SF | Small MF | Large MF | Mobile
area: 0-1499 | 1500-2499 | 2500-3999 | 4000+
income: Low Income (<40,000) | Moderate Income (40,000-99,999) | High Income (>100,000)
climate_zone: Cold | Hot-Dry | Hot-Humid | Marine | Mixed-Dry | Mixed-Humid | Very Cold
heating_efficiency: Low Htg Eff | Medium Htg Eff | High Htg Eff | None/Shared Heating
cooling_type: Heat Pump | High Eff AC | Low Eff AC | Room AC | None
vintage: <1960 | 1960-2000 | >2000
insulation_level: Good Insulation | Average Insulation | Poor Insulation
has_solar: Yes | No

# Defaults to ("Natural Gas","0-1499","Low Income","1960-2000","Low Htg Eff","Good Insulation")
# for PG&E using default tariffs for a zip code in Basleine Territory S if all are empty
"""

gen_electric_bills, _ = genability_costs(elecTariff,gasTariff,state,utility,gas_utility,segment,upgrade)
hack_electric_bills, annual_gas_bills = genability_costs_hack(elecTariff,gasTariff,state,utility,gas_utility,segment,upgrade)


In [ ]:

length = len(hack_electric_bills[0]["distribution"])
gen_length = len(gen_electric_bills[0]["distribution"])
print(f"""
{length} building(s), {gen_length} of which were sampled for genability API cost calculation

Annual Electric Bills:""")

for gen_bill, hack_bill in zip(gen_electric_bills,hack_electric_bills):
    print(f"  {gen_bill['Name']} with {'upgrade measure '+str(gen_bill['Upgrade']) if gen_bill['Upgrade'] else 'baseline profile'}: ${gen_bill['average']:,.2f} ± {gen_bill['moe']:,.2f}, Hack result is ${hack_bill['average']:,.2f} ± {hack_bill['moe']:,.2f}")

print("Annual Gas Bills:")
for bill in annual_gas_bills:
    print(f"  {bill['Name']} with {'upgrade measure '+str(bill['Upgrade']) if bill['Upgrade'] else 'baseline profile'}: ${bill['average']:,.2f} ± {bill['moe']:,.2f}")

print("Total Bills:")
for electric_bill, gas_bill in zip(hack_electric_bills,cycle(annual_gas_bills)):
    print(f"  {'Upgrade measure '+str(electric_bill['Upgrade']) if electric_bill['Upgrade'] else 'Baseline Profile'} for {electric_bill['Name']} electric tariff: ${electric_bill['average']+gas_bill['average']:,.2f} ± {electric_bill['moe']+gas_bill['moe']:,.2f}")

In [ ]:
import plotly.express as px
import pandas as pd

# flatten with labels
flat = [
    {"Bills": v_elec+v_gas, "Tariff+Upgrade": f"{d_elec['Name']}, {d_elec['Upgrade']}"}
    for d_elec,d_gas in zip(hack_electric_bills,cycle(annual_gas_bills))
    for v_elec,v_gas in zip(d_elec["distribution"],d_gas["distribution"])
]

df = pd.DataFrame(flat)
fig = px.box(df, x="Tariff+Upgrade", y="Bills")
fig.update_layout(title=f"""Total Bills for {', '.join([seg for seg in segment.values() if seg])} in {state}<br>using electricity from {utility} and gas from {gas_utility}""")
fig.show()

In [ ]:
import geopandas as gpd
import polars as pl
import matplotlib.pyplot as plt
import us

buildings_path = "cleaned_resstock_data/full_data_with_utility.csv"
states_path = "cleaned_resstock_data/shapefiles/tl_2024_us_state.shp"
# IOU_path = "cleaned_resstock_data/shapefiles/Gas Territories Fixed.shp"
IOU_path = "cleaned_resstock_data/shapefiles/Electric Territory IOU.shp"
to_fips = us.states.mapping("abbr", "fips")

# Load layers
buildings = pl.read_csv(buildings_path)
states    = gpd.read_file(states_path)
services  = gpd.read_file(IOU_path)

# Specify savings comparison
comp = ("Upgrade Only",3)
comp = ("Upgrade+Tariff Switch", 3, elecTariff[1])

if comp[0] == "Upgrade Only":
    electric_from = pl.from_dict([h for h in hack_electric_bills if h["Upgrade"]==0][0]).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    try:
        electric_to = pl.from_dict([h for h in hack_electric_bills if h["Upgrade"]==comp[1]][0]).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    except Exception:
        raise Exception("Upgrade chosen not processed") from None
    
    electric_savings = electric_to.with_columns([
        (pl.col("distribution") - electric_from["distribution"]).alias("distribution")
    ])

    gas_from = pl.from_dict([h for h in annual_gas_bills if h["Upgrade"]==0][0]).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    gas_to = pl.from_dict([h for h in annual_gas_bills if h["Upgrade"]==comp[1]][0]).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    gas_savings = gas_to.with_columns([
        (pl.col("distribution") - gas_from["distribution"]).alias("distribution")
    ])
    
    total_savings = electric_savings.with_columns([
        (pl.col("distribution") + gas_savings["distribution"]).alias("distribution")
    ])
elif comp[0] == "Upgrade+Tariff Switch":
    electric_from = pl.from_dict([h for h in hack_electric_bills if h["Upgrade"]==0][0], strict=False).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    try:
        electric_to = pl.from_dict([h for h in hack_electric_bills if h["Upgrade"]==comp[1] and h["tariff"]==comp[2]][0], strict=False).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    except Exception:
        raise Exception("Upgrade or tariff chosen are not processed") from None
    
    electric_savings = electric_to.with_columns([
        (pl.col("distribution") - electric_from["distribution"]).alias("distribution")
    ])

    gas_from = pl.from_dict([h for h in annual_gas_bills if h["Upgrade"]==0][0], strict=False).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    gas_to = pl.from_dict([h for h in annual_gas_bills if h["Upgrade"]==comp[1]][0], strict=False).select(["ids","distribution"]).rename({"ids":"bldg_id"})
    gas_savings = gas_to.with_columns([
        (pl.col("distribution") - gas_from["distribution"]).alias("distribution")
    ])
    
    total_savings = electric_savings.with_columns([
        (pl.col("distribution") + gas_savings["distribution"]).alias("distribution")
    ])

# Filter states and set CRS
states = states[states["STATEFP"]==to_fips[state]]
buildings = buildings.filter((pl.col("in.state")==state)&(pl.col("bldg_id").is_in(total_savings["bldg_id"]))).join(total_savings,on="bldg_id")
common_crs   = states.crs
services     = services.to_crs(common_crs)

# Prepare the IOU layer
if "Electric" in IOU_path:
    services = services[services["UtilitySta"]==state]
else:
    services = services[services["State"]==state]

# Build a GeoDataFrame of buildings
buildings = buildings.to_pandas()
buildings = gpd.GeoDataFrame(
    buildings,
    geometry=gpd.points_from_xy(buildings["in.weather_file_longitude"], buildings["in.weather_file_latitude"]),
    crs="EPSG:4326"
)

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

# State boundaries
states.boundary.plot(ax=ax, edgecolor="black", linewidth=1)

# Service territories
services.plot(
    ax=ax,
    column="Company",
    categorical=True,
    legend=True,
    linewidth=0.5,
    alpha=0.6,
    edgecolor="grey",
    cmap="tab20"
)

# Buildings plot
# group by geometry to count duplicates
costs = buildings.groupby(buildings["in.county_name"]).agg({"distribution":"mean","geometry":"first"})
costs["count"] = buildings.groupby(buildings["in.county_name"]).size().values
costs = gpd.GeoDataFrame(costs, geometry="geometry", crs=buildings.crs)

costs.plot(
    ax=ax,
    color="red",
    markersize=costs["count"]/costs["count"].max()*500,
    label="Building Representative Points",
    alpha=0.5
)

for x, y, c, n in zip(costs.geometry.x, costs.geometry.y, costs["distribution"], costs["count"]):
    ax.text(x, y, f"${round(c)}, n={n}", fontsize=8, ha="center", va="center", color="black")

ax.set_title(f"Buildings & Service Territories in {state} (n={costs['count'].sum()})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats as st
## Correlation of bill impact with consumption
print("Correlation of bill impact with consumption (close to -1 more savings from more consumption, 0 none, close to 1 increased cost from increased consumption)")
print(buildings["distribution"].corr(buildings["out.electricity.total.energy_consumption.kwh"]))

## Correlation of bill impact with building size
groups = [group["distribution"].values for _, group in buildings.groupby("area")]
f_stat, p_val = st.f_oneway(*groups)
print("Correlation to building size (close to 1 means strong correlation)")
print(p_val)
## Correlation of bill impact with building size
groups = [group["distribution"].values for _, group in buildings.groupby("vintage")]
f_stat, p_val = st.f_oneway(*groups)
print("Correlation to building vintage (close to 1 means strong correlation)")
print(p_val)
## Correlation of bill impact with income level
groups = [group["distribution"].values for _, group in buildings.groupby("income")]
f_stat, p_val = st.f_oneway(*groups)
print("Correlation to income level (close to 1 means strong correlation)")
print(p_val)
## Correlation of bill impact with location
groups = [group["distribution"].values for _, group in buildings.groupby("in.county_name")]
f_stat, p_val = st.f_oneway(*groups)
print("Correlation to location by county (close to 1 means strong correlation)")
print(p_val)
## Correlation of bill impact with insulation
groups = [group["distribution"].values for _, group in buildings.groupby("insulation_level")]
f_stat, p_val = st.f_oneway(*groups)
print("Correlation to location by insulation level (close to 1 means strong correlation)")
print(p_val)

In [ ]:
buildings_bin = buildings.drop("geometry", axis=1)
bin_len = 50
bin_edges = list(range(int(buildings_bin["distribution"].min()//bin_len * bin_len - bin_len), int(buildings_bin["distribution"].max()//bin_len * bin_len + bin_len), bin_len))
buildings_bin["bin"] = pd.cut(buildings_bin["distribution"], bins=bin_edges)

# count per bin
binned = buildings_bin["bin"].value_counts().sort_index().reset_index()
binned.columns = ["bin", "count"]
binned["bin"] = bin_edges[:-1]

# plot it
fig = px.bar(binned, x="bin", y="count")
fig.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

order_by_size = {
    "building_type": ["SF", "Small MF", "Large MF", "Mobile"],
    "area": ["0-1499", "1500-2499", "2500-3999", "4000+"],
    "income": ["Low Income", "Moderate Income", "High Income"],
    "heating_efficiency": ["None/Shared Heating", "Low Htg Eff", "Medium Htg Eff", "High Htg Eff"],
    "cooling_type": ["None", "Room AC", "Low Eff AC", "High Eff AC", "Heat Pump"],
    "vintage": ["<1960", "1960-2000", ">2000"],
    "insulation_level": ["Poor Insulation", "Average Insulation", "Good Insulation"]
}

# list of ordered columns and their orderings
ordered_cols = list(order_by_size.keys())
ordered_orders = list(order_by_size.values())


# encode
encoder = OrdinalEncoder(categories=ordered_orders)
encoded = encoder.fit_transform(buildings[ordered_cols])

# build df
encoded_df = pd.DataFrame(encoded, columns=ordered_cols, index=buildings.index)
encoded_df["distribution"] = buildings["distribution"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import shap

X = encoded_df.drop(columns=["distribution"])
y = encoded_df["distribution"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar")
shap.summary_plot(shap_values, X_test)